# Feature Engineering

## Objective
Transform raw data into meaningful features based on EDA insights to improve model performance.

## Load Data

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "../data/raw/"

DATA_PATH = "../data/raw/"
df = pd.read_csv(DATA_PATH + "train.csv")

df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


## Feature Engineering

### 1. Tenure Features

In [42]:
# Tenure buckets
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['new', 'early', 'mid', 'loyal']
)

# Early churn risk flag
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)

#### Why
- Early customers have significantly higher churn
- Captures lifecycle stage explicitly

### 2. Pricing Features

In [43]:
# Normalize total charges (avoid leakage carefully)
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)

# High charge flag
df['high_charges'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

#### Why
- High monthly charges correlate with churn
- Average spend captures customer value

### 3. Contract Features

In [44]:
# Ordinal encoding for contract
contract_map = {
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
}

df['contract_type'] = df['Contract'].map(contract_map)

#### Why
- Strongest categorical predictor
- Represents customer commitment level

### 4. Service Features

In [45]:
service_cols = [
    'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport'
]

# Count how many services a customer has
df['num_services'] = (df[service_cols] == 'Yes').sum(axis=1)

# No service flag
df['no_service'] = (df['num_services'] == 0).astype(int)

#### Why
- More services → lower churn
- Captures overall engagement level

### 5. Payment Behavior

In [46]:
# Auto payment flag
df['is_auto_payment'] = df['PaymentMethod'].isin([
    'Bank transfer (automatic)',
    'Credit card (automatic)'
]).astype(int)

# Risky payment method
df['is_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

#### Why
- Payment behavior strongly correlates with churn
- Captures reliability & engagement

### 6. Interaction Features

In [47]:
# High value but low tenure (risky customers)
df['high_value_new'] = (
    (df['MonthlyCharges'] > df['MonthlyCharges'].median()) &
    (df['tenure'] <= 12)
).astype(int)

#### Why
- Combines pricing + lifecycle
- Captures high-risk segment identified in EDA

In [48]:
drop_cols = ['id', 'Contract', 'PaymentMethod']
df = df.drop(columns=drop_cols)

In [49]:
# Encode first
df_encoded = pd.get_dummies(df, drop_first=True)

# Then split X and y
X = df_encoded.drop(columns=['Churn_Yes'])  # after encoding
y = df_encoded['Churn_Yes']

### Feature Testing

In [52]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# Encode
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop(columns=['Churn_Yes'])
y = df_encoded['Churn_Yes']

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def train_eval(features):
    model = LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1)
    model.fit(X_train[features], y_train)
    preds = model.predict_proba(X_val[features])[:, 1]
    return roc_auc_score(y_val, preds)

current_features = list(X.columns)
best_score = train_eval(current_features)

improved = True

while improved:
    improved = False
    
    for col in tqdm(current_features):
        trial_features = [f for f in current_features if f != col]
        score = train_eval(trial_features)
        
        if score >= best_score:
            best_score = score
            current_features = trial_features
            improved = True
            break

# Final output
print("Final Score:", round(best_score, 5))
print("Number of Features:", len(current_features))
print("Selected Features:")
print(current_features)

100%|███████████████████████████████████████████████████████████████████████████████████| 25/25 [00:23<00:00,  1.07it/s]

Final Score: 0.91632
Number of Features: 25
Selected Features:
['SeniorCitizen', 'tenure', 'TotalCharges', 'avg_monthly_spend', 'contract_type', 'num_services', 'is_auto_payment', 'is_electronic_check', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'OnlineSecurity_Yes', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'PaperlessBilling_Yes', 'tenure_group_early', 'tenure_group_mid']


### Feature Selection Strategy (Greedy Elimination)

At this stage, the goal is to refine the feature set before building the final model.

Instead of relying solely on feature importance, I applied a **greedy feature elimination approach** using a simple LightGBM model. The idea is to iteratively remove features that do not contribute meaningful predictive power.

#### Methodology

1. Start with all engineered and encoded features  
2. Train a baseline LightGBM model and record performance (ROC-AUC)  
3. Iteratively:
   - Remove one feature at a time  
   - Retrain the model on the reduced feature set  
   - Evaluate performance on a validation split  
4. If removing a feature **does not degrade performance**, it is permanently excluded  
5. Repeat until no further improvements can be made  

This results in a smaller, more efficient set of features without sacrificing model performance.


## Apply the Feature Engineering and Selection to the Dataset

In [53]:
def feature_engineering(df):
    # Tenure
    df['tenure_group'] = pd.cut(
        df['tenure'],
        bins=[0, 12, 24, 48, 72],
        labels=['new', 'early', 'mid', 'loyal']
    )
    df['is_new_customer'] = (df['tenure'] <= 12).astype(int)

    # Pricing
    df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)
    df['high_charges'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

    # Contract
    contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
    df['contract_type'] = df['Contract'].map(contract_map)

    # Services
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['num_services'] = (df[service_cols] == 'Yes').sum(axis=1)
    df['no_service'] = (df['num_services'] == 0).astype(int)

    # Payment
    df['is_auto_payment'] = df['PaymentMethod'].isin([
        'Bank transfer (automatic)', 'Credit card (automatic)'
    ]).astype(int)
    df['is_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

    # Interaction
    df['high_value_new'] = (
        (df['MonthlyCharges'] > df['MonthlyCharges'].median()) &
        (df['tenure'] <= 12)
    ).astype(int)

    return df

Task was destroyed but it is pending!
task: <Task pending name='Task-488' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-489' coro=<Kernel.shell_main() running at /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/theojeremiah/.pyenv/versions/3.11.9/lib/python3.11/tokenize.py:529: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)
Task was destroyed but it is pending!
task: <Task pending name='Task-489' coro

In [54]:
DATA_PATH = "../data/raw/"

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

print(train.shape, test.shape)
train.head()

(594194, 21) (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [55]:
train = feature_engineering(train)
test = feature_engineering(test)

In [56]:
train_encoded = pd.get_dummies(train, drop_first=True)
test_encoded = pd.get_dummies(test, drop_first=True)

# Align columns (VERY IMPORTANT)
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

In [57]:
X = train_encoded.drop(columns=['Churn_Yes'])
y = train_encoded['Churn_Yes']

In [60]:
selected_features = [
    'SeniorCitizen', 'tenure', 'TotalCharges', 'avg_monthly_spend',
    'contract_type', 'num_services', 'is_auto_payment',
    'is_electronic_check', 'Partner_Yes', 'Dependents_Yes',
    'PhoneService_Yes', 'MultipleLines_No phone service',
    'MultipleLines_Yes', 'InternetService_Fiber optic',
    'OnlineSecurity_Yes', 'DeviceProtection_Yes',
    'TechSupport_No internet service', 'TechSupport_Yes',
    'StreamingTV_No internet service', 'StreamingTV_Yes',
    'StreamingMovies_No internet service', 'StreamingMovies_Yes',
    'PaperlessBilling_Yes', 'tenure_group_early', 'tenure_group_mid'
]

X_final = X[selected_features]

In [61]:
X_test_final = test_encoded[selected_features]

In [62]:
# Train
train_final = X_final.copy()
train_final['Churn'] = y
train_final.to_csv("../data/processed/train_final.csv", index=False)

# Test
X_test_final.to_csv("../data/processed/test_final.csv", index=False)